# Classification

* Binary
    - The input either belongs to a class or not
* Multiclass
    - The input belongs to (only) 1 among multiple classes
* Multilabel
    - The input can belong to more than 1 class at the same time

---
In Computational Vision, it's more common using 4x4 images in **batch of 32**
 - That results into a 3x224x224 matrix (3 color per pixel, 224x224 pixel²)
 - The more recommended batch size for every ML project is at most 32
     - That is, the model should learn form 32 images at a time
     - Even if it is not 32, a multiple of 8 is more efficient
- The matrix is: `[` `batch_size`, `colour_channels`, `width`, `height` `]`



# Architecture for Binary x Multiclass Classification

Hyprparameter               | Binary                     | Multiclass       |
| ---------                 | ---                        | ---              |
| Input Layer Shape         | Nº of Features             | Same             |
| Hidden Layers             | Problem-specific 1-oo      | Same             |
| Nerons per Hidden Layer   | Problem-spec. norm. 10-512 | Same             |
| Output Layer Shape        |  1                         | 1 per class      |
| Hidden Layer Activation   | Usually ReLu               | Same             |
| Output Activation         | Sigmoid                    | SoftMax          |
| Loss Function             | Binary Crossentropy        | Cross Entropy    |
| Optimizer                 | SGD, Adam                  | Same             |

In [ ]:
# Get Classification Data
import sklearn.datasets as sd

# Making samples
N_SAMPLES = 1000
RND_SEED = 0
TEST_SIZE = 0.2

In [ ]:
import torch
# Randomness
torch.manual_seed(RND_SEED)
# device-agnostic
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Pytorch Version:", torch.__version__)

# A toy dataset (small to experiment, but enough to learn)
NOISE = 0.03
X, y = sd.make_circles(N_SAMPLES, noise=NOISE, random_state=RND_SEED)
# Split dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE
                                                    , random_state=RND_SEED)


print(len(X_train), len(X_test), len(y_test), len(y_test))
# using pytorch default float instead of numpy's
X_train = torch.from_numpy(X_train).type(torch.float).to(device)
y_train = torch.from_numpy(y_train).type(torch.float).to(device)
X_test = torch.from_numpy(X_test).type(torch.float).to(device)
y_test = torch.from_numpy(y_test).type(torch.float).to(device)

# Verifying the shapes
print("Shapes =", X.shape, "-->", y.shape)
print("----")
print("X = \n", X[:5])
print("----")
print("y = \n", y[:5])

IN_FEAT = 2
OUT_FEAT = 1
if len(X.shape) > 1:
    IN_FEAT = X.shape[1]
if len(y.shape) > 1:
    OUT_FEAT = y.shape[1]
print("----")
print("Input Features:", IN_FEAT)
print("Output Features:", OUT_FEAT)

In [ ]:
# Make a dataframe with pandas
import pandas as pd

if IN_FEAT == 2:
    df = pd.DataFrame({"X1": X[:, 0], "X2": X[:, 1], "label": y})
    print(df.label.value_counts())
    print("---")
    print(df)
else:
    print("hard to plot")

In [ ]:
# Vizualizing the data
import matplotlib.pyplot as plt

if IN_FEAT == 2:
    plt.scatter(x=X[:, 0],    # horizontal
                y=X[:, 1],    # vertical
                c=y,          # coloring based on the label
                cmap=plt.cm.RdYlBu          # colormap
    );

In [ ]:
import torch.nn as nn
# must be used before any operation
torch.manual_seed(RND_SEED)

class MyLinearBinaryClassifier(nn.Module):
    INTER_FEAT = 128
    def __init__(self):
        super().__init__()
        # layers
        self.layer_1 = nn.Linear(in_features=IN_FEAT,
                                 out_features=self.INTER_FEAT,
                                 device=device
                                 )
        self.layer_2 = nn.Linear(in_features=self.INTER_FEAT,
                                 out_features=self.INTER_FEAT*2,
                                 device=device
                                 )
        self.layer_3 = nn.Linear(in_features=self.INTER_FEAT*2,
                                 out_features=self.INTER_FEAT,
                                 device=device
                                 )
        self.layer_4 = nn.Linear(in_features=self.INTER_FEAT,
                                 out_features=OUT_FEAT,
                                 device=device
                                 )

    def forward(self, x: torch.Tensor):
        # putting them in a single line is faster
        return  self.layer_4(
                self.layer_3(
                self.layer_2(
                self.layer_1(
                    x
                )
                )
                )
                )

model = MyLinearBinaryClassifier()
print(next(model.parameters()))
# print(model.state_dict())

# # will automatically use the layers on forward
# # it could also be used inside the subclass
# seq_model = nn.Sequential(
#     nn.Linear(in_features=IN_FEAT, out_features=model.INTER_FEAT),
#     nn.Linear(in_features=model.INTER_FEAT, out_features=OUT_FEAT)
# ).to(device)

# print(next(seq_model.parameters()))

In [ ]:
# uses nn.Sigmoid built in (being more numerically stable)
# while nn.BCELoss requires the user to send them through the sigmoid prior
loss_fn = nn.BCEWithLogitsLoss()
LR = 0.1
optimizer = torch.optim.SGD(params=model.parameters(), lr=LR)

def accuracy_fn(y_true: torch.Tensor, y_pred: torch.Tensor):
    # how many correct guesses
    correct = torch.eq(y_true, y_pred).sum().item()
    # percent of correct
    acc = (correct / len(y_pred)) * 100
    return acc
#

# Note:
* The model output will be raw **logits**
    * Refering to unscaled probabilities
* It must go:
    * `raw logits` -> `prediction probabilities` -> `prediction labels`
* The activation function turn the logits into probabilities, an them, by rounding them into either 0 or 1, they ecome labels

In [ ]:
# Training Loop
torch.manual_seed(RND_SEED)

EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()

    # get the logits, squeezed in a single verctor
    y_logits = model(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits))

    # get loss and accuracy
    train_loss = loss_fn(y_logits,
                         y_train) # does it own activation
    train_acc = accuracy_fn(y_true=y_train, y_pred=y_pred)

    # Optimizer zero grad
    optimizer.zero_grad()

    # Backpropagation
    train_loss.backward()

    # gradiet descent
    optimizer.step()

    model.eval()
    with torch.inference_mode():
        test_logits = model(X_test).squeeze()
        test_pred = torch.round(torch.sigmoid(test_logits))
        test_loss = loss_fn(test_logits, y_test)
        test_acc = accuracy_fn(y_true=y_test, y_pred=test_pred)
        if epoch % 10 == 0 or epoch == (EPOCHS - 1):
            print("Epoch:", epoch,
                  "- Train Loss:", train_loss.item(),
                  "- Train Acc:", train_acc,
                  "- Test Loss:", test_loss.item(),
                  "- Test Acc:", test_acc
                  )

In [ ]:
# to use a non-std library
import requests
from pathlib import Path

# download the function if it isn't already there
if Path("helper_functions.py").is_file():
    print("helper functions already imported")
else:
    print("downloading helper_funtions.py")
    req = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")
    with open("helper_functions.py", 'wb') as f:
        f.write(req.content)

In [ ]:
from helper_functions import plot_predictions, plot_decision_boundary

plt.figure(figsize=(12, 6))
plt.subplot(1,2,1) # to compare training and test sets
plt.title("Train")
plot_decision_boundary(model, X_train, y_train)
plt.subplot(1,2,2)
plt.title("Test")
plot_decision_boundary(model, X_test, y_test)

# Non-Linearity

The above model try to represent a non-linear relation through a simple line, so, no matter how the layers, units or epochs are increased, it will never acurately represent the relationships between the inputs and outputs (as there is no line that can)

## Ways to Improve a Model (from a model's perspective)
* Add more training epochs
* Add more hidden layers
* Add more units (neurons) per hidden layer
* Change the Activation Function
* Change the Learning Rate
    * Either by increasing it to allow faster iteration
    * Or reducing it to achieve more prcise results
* Change the Loss Function
---
All of the above refer to `Hyprparameters` (parameters that the engineer can change, rather than the ones defined through training)

In [ ]:
torch.manual_seed(RND_SEED)

class MyBinaryClassifier(nn.Module):
    INTER_FEAT = 10
    def __init__(self):
        super().__init__()
        self.act_fn = nn.ReLU()
        self.layers = nn.Sequential(
            nn.Linear(
                in_features=IN_FEAT,
                out_features=self.INTER_FEAT,
                device=device
            ),
            # Using a non-linear function to
            # allow representing non-linear relationships
            self.act_fn,
            nn.Linear(
                in_features=self.INTER_FEAT,
                out_features=self.INTER_FEAT*2,
                device=device
            ),
            self.act_fn,
            nn.Linear(
                in_features=self.INTER_FEAT*2,
                out_features=self.INTER_FEAT,
                device=device
            ),
            self.act_fn,
            nn.Linear(
                in_features=self.INTER_FEAT,
                out_features=OUT_FEAT,
                device=device
            ),

        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

model_v2 = MyBinaryClassifier()
print(model_v2)


In [ ]:
torch.manual_seed(RND_SEED)
EPOCHS = 1001

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_v2.parameters(), lr=0.1)
def logits_to_preds(x: torch.Tensor):
    return torch.round(torch.sigmoid(x))

model_v2.to(device)

for epoch in range(EPOCHS):
    model_v2.train()
    # forward pass
    train_logits = model_v2(X_train).squeeze()
    train_preds = logits_to_preds(train_logits)
    # calculate error
    train_loss = loss_fn(train_logits, y_train)
    train_acc = accuracy_fn(y_true=y_train, y_pred=train_preds)
    # zero grad
    optimizer.zero_grad()
    # backpropagation
    train_loss.backward()
    # gradient descent
    optimizer.step()
    ## Teste
    model.eval()
    with torch.inference_mode():
        test_logits = model_v2(X_test).squeeze()
        test_preds = logits_to_preds(test_logits)
        test_loss = loss_fn(test_logits, y_test)
        test_acc = accuracy_fn(y_true=y_test, y_pred=test_preds)
        if epoch % 100 == 0:
            print(f"Epoch {epoch}:\n - Training Loss:  {train_loss:.5f}\n - Training Acc:  {train_acc:.2f}%\n - Test Loss:      {test_loss:.5f}\n - Test Acc:      {test_acc:.2f}%")


In [ ]:
from helper_functions import plot_predictions, plot_decision_boundary

plt.figure(figsize=(12, 6))
plt.subplot(1,2,1) # to compare training and test sets
plt.title("Train")
plot_decision_boundary(model_v2, X_train, y_train)
plt.subplot(1,2,2)
plt.title("Test")
plot_decision_boundary(model_v2, X_test, y_test)

## Evaluating Classification

# Multi-Class Classification


In [ ]:
# Toy multiclass data
import torch
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyprparameters for data creation
TEST_SIZE = 0.2
RND_SEED = 42
N_SAMPLES = 1000
N_FEATURES = 2
N_CLASSES = 4
DETOUR = 1.5

torch.manual_seed(RND_SEED)
if device == 'cuda':
    torch.cuda.manual_seed(RND_SEED)

X_blob, y_blob = make_blobs(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    centers=N_CLASSES,
    cluster_std=DETOUR,
    random_state=RND_SEED
)

print(X_blob.shape, y_blob.shape)

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X_blob, y_blob, test_size=TEST_SIZE, random_state=RND_SEED)

X_train_3 = torch.from_numpy(X_train_3).type(torch.float).to(device)
y_train_3 = torch.from_numpy(y_train_3).type(torch.LongTensor).to(device)
X_test_3 = torch.from_numpy(X_test_3).type(torch.float).to(device)
y_test_3 = torch.from_numpy(y_test_3).type(torch.LongTensor).to(device)
# X_train_3.requires_grad = True
# y_train_3.requires_grad = True
# X_test_3.requires_grad = True
# y_test_3.requires_grad = True

# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 6))
# plt.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob);

In [ ]:
torch.manual_seed(RND_SEED)
class MyMultiClassifier(nn.Module):
    def __init__(self, in_feat: int, out_feat: int, hidden_units: int=8):
        """
        Initializes a multi-class classifier

        Args:
            in_feat: Int
                Number of input features/values the model receives per prediction
            out_feat: Int
                Number of possible classed the model must predict
            hidden_layers: Int
                Number of neurons per intermediate layer
        Returns:
            ...
        Raises:
            ...
        """
        super().__init__()
        self.act_fn = nn.Identity()
        self.layers = nn.Sequential(
            nn.Linear(
                in_features=in_feat,
                out_features=hidden_units,
            ),
            self.act_fn,
            nn.Linear(
                in_features=hidden_units,
                out_features=hidden_units,
            ),
            self.act_fn,
            nn.Linear(
                in_features=hidden_units,
                out_features=out_feat,
            )
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

model_v3 = MyMultiClassifier(2, 4).to(device)
print(model_v3)
print(model_v3.state_dict())

In [ ]:
torch.manual_seed(RND_SEED)
# Train Loop

# Loss function and optimizer
loss_fn_v3 = nn.CrossEntropyLoss()
optimizer_v3 = torch.optim.Adam(params=model_v3.parameters(), lr=0.01)

EPOCHS = 101

for ep in range(EPOCHS):
    model_v3.train()

    train_logits = model_v3(X_train_3).squeeze()
    # uses argmax to get the index of the most probable class
    train_preds = torch.softmax(train_logits, dim=1).argmax(dim=1)

    train_loss = loss_fn_v3(train_logits, y_train_3)
    train_acc = accuracy_fn(y_true=y_train_3,
                            y_pred=train_preds)

    optimizer_v3.zero_grad()
    train_loss.backward()
    optimizer_v3.step()

    model_v3.eval()
    with torch.inference_mode():
        test_logits = model_v3(X_test_3).squeeze()
        test_preds = torch.softmax(test_logits, dim=1).argmax(dim=1)

        test_loss = loss_fn_v3(test_logits, y_test_3)
        test_acc = accuracy_fn(y_true=y_test_3,
                                y_pred=test_preds)
        if ep % 10 == 0:
            print(f"Epoch: {ep}\n - Train Loss: {train_loss:.4f}\n - Test  Loss: {test_loss:.4f}\n - Train Acc: {train_acc:.2f}%\n - Test  Acc: {test_acc:.2f}%")

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1,2,1)
plt.title("Train")
plot_decision_boundary(model_v3, X_train_3, y_train_3)
plt.subplot(1,2,2)
plt.title("Test")
plot_decision_boundary(model_v3, X_test_3, y_test_3)

# Classification Metrics

* Accuracy
    * How many, out of all, the model agrees with the right result
    * Misleading for imbalanced classes
        ```math
                tp + tn
            ------------------
            tp + tn + fp + fn
        ```
    * `sklearn.metrics.accuracy_score()`
    * `torchmetrics.Accuracy()`

* Precision
    * How many times the model nailed the positive result, out of all the times **the model said the result was positive**
        ```math
               tp
            --------
            tp + fp
        ```
    * Has a tradeoff with recall
    * `sklearn.metrics.precision_score()`
    * `torchmetrics.Precision()`
* Recall
    * How many times the model nailed the positive result, out of all the times **it should have**
        ```math
               tp
            --------
            tp + fn
        ```
    * Has a tradeoff with precision
    * `torchmetrics.Recall()`
    * `sklearn.metrics.recall_score()`
* F1-Score
    * Combines Precision and Recall
        ```math
            2 * precision * recall
            ----------------------
              precision + recall
        ```
    * `sklearn.metrics.f1_score()`
    * `torchmetrics.F1Score()`
* Confusion Matrix
    * `torchmetrics.ConfusionMatrix()`
* Classification Report

In [ ]:
!pip install torchmetrics
import torchmetrics